# 07 — Sharding simulation: meshes, PartitionSpecs, all-reduce cost

**Learning goal.** Build the conceptual model of SPMD on TPUs: a **mesh** of devices, **PartitionSpecs** that map tensor dims to mesh axes, and the **all-reduce** cost that buys you the result.

**What you'll observe.**
- 1D and 2D meshes built with the simulator's `Mesh` type.
- Tensors partitioned with various `PartitionSpec`s — shard counts and per-shard shapes.
- A plot of all-reduce time as a function of chip count for a fixed payload.

**Knobs to tune.**
- Mesh shape: `make_1d_mesh(8)` vs `make_2d_mesh(2, 4)`.
- `PartitionSpec` values: replicated, sharded on one axis, sharded on two axes.
- `payload_bytes` and `ici_bandwidth_bytes_s` in the all-reduce model.

**Failure modes to watch.**
- A PartitionSpec rank that doesn't match the tensor rank raises immediately.
- Sharding along an axis with size > tensor dim quietly clamps shard dim to 1.
- Adding more chips lowers per-chip work but raises collective cost — there's a sweet spot.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## 1D mesh — pure data parallel

In [ ]:
from cloud_tpu_lab.src.sharding.mesh import (
    Mesh, PartitionSpec, make_1d_mesh, make_2d_mesh,
)
from cloud_tpu_lab.src.sharding.partitioner import partition_tensor

mesh1d = make_1d_mesh(8, name="data")
print("1D mesh:", mesh1d)
print("n_devices:", mesh1d.n_devices)

# Shard a (2048, 768) embedding table along the 'data' axis.
shards = partition_tensor(
    logical_shape=(2048, 768),
    spec=PartitionSpec(("data", None)),
    mesh=mesh1d,
)
print("n shards:", len(shards))
print("first shard shape:", shards[0].shard_shape)
print("first shard sharded_axes:", shards[0].sharded_axes)

## 2D mesh — DP x TP

In [ ]:
mesh2d = make_2d_mesh(2, 4, names=("data", "model"))
print("2D mesh:", mesh2d)
print("axis size 'data':", mesh2d.axis_size("data"))
print("axis size 'model':", mesh2d.axis_size("model"))

# Shard a Linear weight (in=1024, out=2048) along 'model' (output dim).
shards = partition_tensor(
    logical_shape=(1024, 2048),
    spec=PartitionSpec((None, "model")),
    mesh=mesh2d,
)
print("Linear sharded by output on model axis:")
print("  n shards:", len(shards))
print("  shard shape:", shards[0].shard_shape)

## Different PartitionSpecs on the same tensor

In [ ]:
tensor_shape = (1024, 768)

cases = [
    ("replicated",      PartitionSpec((None, None))),
    ("DP rows",         PartitionSpec(("data", None))),
    ("TP cols",         PartitionSpec((None, "model"))),
    ("2D shard",        PartitionSpec(("data", "model"))),
]

print(f"{'spec':<14} {'shards':>6}  {'shard_shape':>16}  {'axes'}")
print("-" * 60)
for label, spec in cases:
    shards = partition_tensor(tensor_shape, spec, mesh2d)
    print(f"{label:<14} {len(shards):>6}  {str(shards[0].shard_shape):>16}  {shards[0].sharded_axes}")

## All-reduce time vs chip count

`all_reduce_time` uses the ring-allreduce bandwidth-optimal model:

  `t ~ link_latency * (N-1)  +  2(N-1)/N * payload_bytes / ICI_bw`

As `N` grows, the latency term grows linearly but the bandwidth term saturates at `2 * payload / ICI_bw`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from cloud_tpu_lab.src.sharding.all_reduce import all_reduce_time
from cloud_tpu_lab.src.tpu_versions.cloud_tpu_catalog import get_spec

spec = get_spec("v5e")
ici_bw_bytes_s = spec.ici_bandwidth_gbps * 1e9

payload_bytes_choices = [1 * 1024,           # 1 KiB — latency-bound
                         1 * 1024 * 1024,    # 1 MiB
                         100 * 1024 * 1024]  # 100 MiB — bandwidth-bound

chip_counts = [1, 2, 4, 8, 16, 32, 64]

fig = plt.figure(figsize=(8, 5))
for payload in payload_bytes_choices:
    times = []
    for n in chip_counts:
        c = all_reduce_time(payload, n, ici_bw_bytes_s)
        times.append(c.sim_duration_s * 1000)
    plt.plot(chip_counts, times, marker="o", label=f"{payload/1024/1024:.2f} MiB")

plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("number of chips")
plt.ylabel("all-reduce time (ms)")
plt.title(f"All-reduce on {spec.code_name}: ring-allreduce alpha+beta*n model")
plt.legend(title="payload")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb07_all_reduce_vs_chips.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Reading the plot

- For small payloads (1 KiB), latency dominates and the line is roughly linear in `N`.
- For large payloads (100 MiB), bandwidth dominates and the line plateaus — adding chips past a point doesn't lower comm time. Per-chip *compute* still shrinks linearly, so total step time still drops, but you start hitting diminishing returns.
- This is the basic intuition behind picking pod size. Notebook 09 turns this into a full scaling plot.

## Exercises

1. Switch to `get_spec("v5p")` (much higher ICI bandwidth). Replot. Where does the latency floor become visible?
2. Sweep `payload_bytes` from 1 KiB to 1 GiB at a fixed `n_chips=16`. Plot allreduce time vs payload on log-log axes. Identify the latency-bound region.
3. Use `all_gather_time` and `reduce_scatter_time` from the same module. For an all-reduce decomposed as reduce-scatter + all-gather, does the sum match `all_reduce_time`?
4. Construct a 3D mesh of shape `(2, 2, 2)` with axes `("data", "model", "pipeline")`. Show two PartitionSpecs that result in the same shard shape but different `sharded_axes`.
5. Why does a `PartitionSpec(("data", "model"))` on a 2D mesh of shape `(2, 4)` produce different shard shape than on a `(4, 2)` mesh?